# LimX HU_D03 mimic training on Kaggle (mjlab)

Trains `Mjlab-Tracking-Flat-LimX-HU-D03`: the HU_D03 humanoid learns to imitate a reference motion from
LAFAN1 (BeyondMimic-style motion tracking), with [mjlab](https://github.com/mujocolab/mjlab) and the
[LimX HU_D03 model](https://github.com/limxdynamics/humanoid-description).

### Before running

1. **Settings → Accelerator → GPU T4 x2.** Not P100: it is older than Turing and mjlab does not support it.
2. **Settings → Internet → On** (needs a phone-verified Kaggle account).
3. For BONES-SEED (`SOURCE = "seed"`, the default): request access at
   [huggingface.co/datasets/bones-studio/seed](https://huggingface.co/datasets/bones-studio/seed), create a
   Hugging Face **read** token, and add it under **Add-ons → Secrets** as `HF_TOKEN` (tick it for this notebook).

### How to run

* **Try it interactively first:** run the cells down to *5. Prepare the motion* and make sure the
  reference replay looks like the clip (feet on the ground, arms and legs moving the right way).
  Interactive sessions stop when the browser is idle, so do not train there.
* **Train with Save Version → Save & Run All (Commit).** That runs the whole notebook in the background
  (up to 12 h) and keeps everything in `/kaggle/working`: motion file, checkpoints, ONNX policy, training curves,
  videos and a zip.
  Training stops itself after `TIME_BUDGET_HOURS` so the steps after it still run.

### Continue training in a new version

Open the notebook, *Add Input → Your Work → Notebooks*, and add **this notebook's previous version**.
The settings cell finds its newest checkpoint under `/kaggle/input` and resumes from it.
Keep `MOTION`, `MEDLEY` and `LINE_RANGE` the same as in that version: that is how the run is recognised.

### Several dances in one policy

List BONES-SEED filenames (or LAFAN1 clip names) in `MEDLEY`. Section 5 retargets each one, turns and shifts it
to start where the previous one ended, blends `BLEND_SECONDS` between them and trains one policy on the joined
clip. The policy follows whichever dance its reference shows, so it can perform any of them.
Training does not start from scratch either: with `WARM_START = True`, a new clip or medley starts from the newest
checkpoint of any earlier run added under Input (for example your single-dance run), as long as `TASK` is the same.
To add a dance later, append it to `MEDLEY`, add the previous version as Input, and run again: it warm-starts from
the previous medley.

### About the model

The real HU_D03 drives its ankles and waist through parallel linkages (two motors per joint pair). Here they are
serial `ankle_pitch/roll` and `waist_roll/pitch` joints, as mjlab does for the Unitree G1, so the policy's targets
must be converted to A/B motor commands on the real robot. LimX does not publish PD gains; they are derived from
the rotor inertias (10 Hz, damping ratio 2).

The reference motion is LAFAN1 as retargeted to the Unitree G1, copied to the HU_D03 joint by joint
(GMR does not support the HU_D03 yet), so foot and hand positions are approximate.
LAFAN1 is licensed CC BY-NC-ND 4.0 (non-commercial).

## 1. Settings

In [ ]:
import json, os, shutil, signal, subprocess
from pathlib import Path

# Output folder in /kaggle/working, saved with each version. Use a new name per experiment to keep outputs apart
# (e.g. "hu_d03_rl" for the ballet run, "hu_d03_rl_medley" for the medley); resume and warm start find runs in any of them.
OUTPUT_NAME = "hu_d03_rl_medley"
WORK = Path("/kaggle/working") / OUTPUT_NAME
CODE = Path("/kaggle/temp/hu_d03_code")    # Installed code, re-created each run.
WORK.mkdir(parents=True, exist_ok=True)
CODE.mkdir(parents=True, exist_ok=True)

NUM_ENVS = 4096            # Use 2048 if the GPU runs out of memory.
MAX_ITERATIONS = 10_000    # Upper bound; the time budget usually stops training first.
TIME_BUDGET_HOURS = 10.5   # Kaggle stops a version at 12 h; leave time for the video and zip.
USE_TWO_GPUS = False       # True splits envs over both T4s (faster, but less tested).
TRAIN = True               # False: skip training; only plot, record and package the newest run
                           # (this session's, or a previous version's added under Input).

# Code versions. These commits were tested with this notebook; "main" gets the latest.
MJLAB_REF = "8bb2bc9e187797689ec82c78c35019afd2fd0376"
DESCRIPTION_REF = "0dd8454384c58991f3e70fd7613be29006d39f86"

# Reference motion. SOURCE picks the dataset:
#   "seed":   BONES-SEED mocap (dances incl. ballet). Needs access on Hugging Face and an HF_TOKEN secret (see top).
#             MOTION is a BONES-SEED filename; "" uses the longest match of SEED_SEARCH (section 1b lists them).
#   "lafan1": LAFAN1 retargeted to the G1, no sign-up. MOTION is a clip name: walk1_subject1, dance1_subject1..3,
#             dance2_subject1..5, run1_subject2, jumps1_subject1, fight1_subject2, ...
SOURCE = "seed"
MOTION = ""
SEED_SEARCH = "ballet"     # All words must appear in a motion's descriptions.
LINE_RANGE = None          # e.g. (1, 900) to use only the first 30 s (30 fps) of a long clip.

# Several dances in one policy: clip names of SOURCE (BONES-SEED below), joined in this order; MOTION is then
# ignored. Section 8 records one video per clip. [] trains on MOTION alone.
MEDLEY = [
  "walk_ff_loop_180_R_002__A046",                              # normal walking (16.9 s)
  "dance_hiphop_hip_hop_iv_R_002__A320",                       # fast arm and leg swings (16.3 s)
  "dance_latino_mambo_chase_180_turn_R_fast_002__A312",        # fast turns (11.1 s)
  "dance_vouge_the_duck_walk_variation_180_R_fast_002__A320",  # deep squat walk (9.3 s)
  "dance_latino_jive_sequence_R_101__A321",                    # kicks, quick footwork (25.4 s)
  "victory_dance_macarena_change_direction_R_loop_001__A321",  # jumps changing direction (28.5 s)
]
BLEND_SECONDS = 1.0        # Transition inserted between two clips of the medley.
WARM_START = True          # No earlier run of this clip/medley under Input: start from the newest checkpoint
                           # of any run there (same TASK) instead of from scratch.
WARM_START_FROM = ""       # "" picks automatically. To choose, give a model_*.pt (or a pattern) under /kaggle/input,
                           # e.g. "/kaggle/input/hu-d03-ballet/model_9999.pt" or "**/hu-d03-ballet/**/model_*.pt".
WARM_START_STD = 0.5       # Raise the warm-start policy's action noise to at least this (a fresh policy starts
                           # at 1.0; a trained one is ~0.2), so it explores the new dances. None keeps it.

TASK = "Mjlab-Tracking-Flat-LimX-HU-D03"
EXPERIMENT = "hu_d03_tracking"
LOG_ROOT = WORK / "logs" / "rsl_rl" / EXPERIMENT
EPISODE_STEPS = 500        # 10 s episodes at 50 Hz.

### 1b. Pick the clip

For BONES-SEED this downloads the metadata (5 MB), lists the motions whose descriptions contain all words of
`SEED_SEARCH` (mirrored copies left out, longest first) and picks `MOTION`. To use another one, copy its
`filename` into `MOTION` in the settings and run both cells again.

In [ ]:
import re
if SOURCE == "seed":
  from huggingface_hub import hf_hub_download
  try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
  except Exception as e:
    print("No HF_TOKEN Kaggle secret (Add-ons → Secrets, then tick it for this notebook):", e)
  import pandas as pd
  meta = pd.read_parquet(hf_hub_download(
    "bones-studio/seed", "metadata/seed_metadata_v004.parquet", repo_type="dataset",
    local_dir=WORK / "motions" / "seed"))
  text_cols = [c for c in meta.columns if c.startswith("content_")]
  text = meta[text_cols].fillna("").astype(str).agg(" ".join, axis=1).str.lower()
  hits = meta[text.apply(lambda t: all(w in t for w in SEED_SEARCH.lower().split()))
              & ~meta["is_mirror"].fillna(False).astype(bool)]
  hits = hits.assign(seconds=hits["move_duration_frames"] / 120).sort_values("seconds", ascending=False)
  print(f"{len(hits)} motions match {SEED_SEARCH!r}")
  display(hits[["filename", "seconds", "category", "content_short_description",
                "content_natural_desc_4"]].head(40))
  if not MOTION and not MEDLEY:
    if hits.empty:
      raise SystemExit(f"Nothing matches {SEED_SEARCH!r}. Try other words (e.g. 'pirouette', 'plie', "
                       f"'arabesque', 'dance') or set MOTION to a filename from the BONES-SEED viewer.")
    MOTION = hits.iloc[0]["filename"]

MOTIONS = list(MEDLEY) or [MOTION]
if SOURCE == "seed":
  # Path of each clip's G1 CSV inside g1.tar.gz. The column name changes between metadata versions
  # (v004 has no move_g1_mujoco_path); without one, the archive is searched for <name>.csv anywhere.
  SEED_PATHS = {}
  for name in MOTIONS:
    rows = meta[meta["filename"] == name]
    if rows.empty:
      raise SystemExit(f"{name!r} is not a BONES-SEED filename (copy it from the table above).")
    row = rows.iloc[0]
    g1_cols = [c for c in meta.columns if "g1" in c.lower() and isinstance(row[c], str) and "/" in row[c]]
    SEED_PATHS[name] = row[g1_cols[0]] if g1_cols else None
    print(f"clip: {name} ({row['move_duration_frames'] / 120:.1f} s)\n  {row['content_natural_desc_4']}")

# Clip name safe for file names and the run folder; a medley is named by a hash of its clip list.
if MEDLEY:
  import hashlib
  key = json.dumps([SOURCE, MOTIONS, BLEND_SECONDS])
  CLIP = f"medley{len(MOTIONS)}_{hashlib.sha1(key.encode()).hexdigest()[:8]}"
else:
  CLIP = re.sub(r"[^A-Za-z0-9_.-]+", "_", MOTION)
# Runs are named after the clip (logs/.../<date_time>_<RUN_NAME>) and only a run of the same clip is resumed.
RUN_NAME = CLIP + (f"_{LINE_RANGE[0]}-{LINE_RANGE[1]}" if LINE_RANGE else "")
print("run name:", RUN_NAME)

# Resume: newest checkpoint of this clip from an earlier version added under "Input".
def newest_input_run():
  ckpts = list(Path("/kaggle/input").glob(f"**/logs/rsl_rl/{EXPERIMENT}/*_{RUN_NAME}/model_*.pt"))
  if not ckpts:
    return None
  ckpts.sort(key=lambda p: (p.parent.name, int(p.stem.split("_")[1])))
  return ckpts[-1].parent

RESUME_RUN = newest_input_run()
if RESUME_RUN is not None:
  dest = LOG_ROOT / RESUME_RUN.name
  if not dest.exists():
    shutil.copytree(RESUME_RUN, dest)
  print("will resume from", dest, "| newest checkpoint:",
        max(dest.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1])).name)
else:
  print(f"no earlier {RUN_NAME} run under /kaggle/input")

# Warm start: otherwise begin from a checkpoint under Input: WARM_START_FROM, or else the newest model_*.pt there
# (an earlier version's logs, or a model you uploaded as a dataset, zipped or not). It must come from the same TASK
# (observation and action sizes must match). It is copied next to this run's logs and loaded by train.
def input_checkpoint():
  root = Path("/kaggle/input")
  if WARM_START_FROM:
    pattern = str(Path(WARM_START_FROM).relative_to(root)) if WARM_START_FROM.startswith("/") else WARM_START_FROM
    found = sorted(root.glob(pattern))
    if not found:
      raise SystemExit(f"WARM_START_FROM matches nothing under {root}. Checkpoints there: "
                       f"{[str(p) for p in root.glob('**/model_*.pt')][:20]}")
    return found[-1]
  ckpts = []
  for p in root.glob("**/model_*.pt"):
    if not re.fullmatch(r"model_\d+\.pt", p.name):
      continue
    # Skip runs of another TASK when their version's settings.json says so.
    settings = next((a / "settings.json" for a in p.parents if (a / "settings.json").exists()), None)
    if settings and json.loads(settings.read_text()).get("TASK") != TASK:
      continue
    ckpts.append(p)
  # Newest run (folders start with the date), then highest iteration.
  ckpts.sort(key=lambda p: (p.parent.name, int(p.stem.split("_")[1])))
  return ckpts[-1] if ckpts else None

INIT_RUN = INIT_CHECKPOINT = None
if RESUME_RUN is None and WARM_START:
  ckpt = input_checkpoint()
  if ckpt is not None:
    # A plain folder name: train's --agent.load-run is a regex and goes through the shell.
    INIT_RUN, INIT_CHECKPOINT = "warm_start_" + re.sub(r"[^A-Za-z0-9_-]+", "_", ckpt.parent.name), ckpt.name
    dest = LOG_ROOT / INIT_RUN
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(ckpt, dest / ckpt.name)
    print("warm start from", ckpt)
if RESUME_RUN is None and INIT_RUN is None:
  print("training from scratch")

# Saved so the cells after training can restore everything if the notebook restarts (see section 7).
(WORK / "settings.json").write_text(json.dumps(
  {k: v for k, v in dict(globals()).items()
   if k.isupper() and isinstance(v, (str, int, float, bool, tuple, list, dict, type(None)))}, indent=1))

## 2. Check the GPU

In [ ]:
out = subprocess.run(
  ["nvidia-smi", "--query-gpu=name,compute_cap,driver_version,memory.total", "--format=csv,noheader"],
  capture_output=True, text=True,
)
if out.returncode != 0:
  raise SystemExit("No GPU. Set Settings → Accelerator → GPU T4 x2.")
print(out.stdout)
name, cap, driver, _ = [s.strip() for s in out.stdout.splitlines()[0].split(",")]
if float(cap) < 7.5:
  raise SystemExit(f"{name} (compute {cap}) is too old for mjlab. Pick GPU T4 x2, not P100.")
CUDA_EXTRA = "cu130" if int(driver.split(".")[0]) >= 580 else "cu126"
NUM_GPUS = len(out.stdout.splitlines())
print("mjlab extra:", CUDA_EXTRA, "| GPUs:", NUM_GPUS)

## 3. Install mjlab and the HU_D03 model (about 3–5 minutes)

Helper functions and environment variables. They are saved to `<OUTPUT_NAME>/nb_helpers.py` so the
restore cell in section 7 can load them again after a restart.

In [ ]:
%%writefile {WORK}/nb_helpers.py
import os, signal, subprocess, threading, time
from pathlib import Path
from IPython.display import Video, display

os.environ["WANDB_MODE"] = "disabled"
os.environ["MUJOCO_GL"] = "egl"
os.environ["MJLAB_WARP_QUIET"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"  # Stream training logs into the notebook.
os.environ["PYTHONFAULTHANDLER"] = "1"  # Print a Python traceback on a crash.
os.environ["HU_D03_DESCRIPTION_DIR"] = str(CODE / "humanoid-description" / "HU_D03_description")

def sh(cmd, cwd=None, max_seconds=None, check=True, log_file=None, on_line=None, on_tick=None):
  """Run a shell command and stream its output into the notebook.

  log_file: write the output there instead of the notebook (the last lines are
  shown if the command fails). on_line(line) sees every output line;
  on_tick(elapsed_s) is called every few seconds from the notebook thread.
  After max_seconds the command gets Ctrl-C (then a kill) and counts as done.
  """
  print("$", cmd, flush=True)
  log = open(log_file, "a") if log_file else None
  proc = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, bufsize=1,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          start_new_session=True)
  sh.last_output = []

  def pump_output():
    for line in proc.stdout:
      if log:
        log.write(line)
      else:
        print(line, end="", flush=True)
      if on_line:
        on_line(line)
      sh.last_output.append(line)
      del sh.last_output[:-200]

  pump = threading.Thread(target=pump_output)
  pump.start()
  start, timed_out = time.time(), False
  while proc.poll() is None:
    if on_tick:
      on_tick(time.time() - start)
    if max_seconds is not None and time.time() - start > max_seconds:
      timed_out = True
      print(f"\n[time budget reached after {max_seconds / 3600:.1f} h: stopping]", flush=True)
      os.killpg(proc.pid, signal.SIGINT)
      try:
        proc.wait(timeout=120)
      except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
      break
    time.sleep(10 if on_tick else 0.5)
  pump.join(timeout=30)
  if on_tick:
    on_tick(time.time() - start)
  if log:
    log.close()
  if check and proc.returncode != 0 and not timed_out:
    if log:
      print("".join(sh.last_output[-60:]))
    raise SystemExit(f"command failed with exit code {proc.returncode}")
  return proc.returncode

def video(out, *extra, show=True):
  """Record a video; returns None instead of failing the notebook. show=False: do not embed it."""
  cmd = (f"python {SCRIPTS / 'record_video.py'} {TASK} --output-file {out} "
         f"--motion-file {MOTION_FILE} " + " ".join(extra))
  if sh(cmd, cwd=WORK, check=False) != 0:
    egl_error = "EGL" in "".join(sh.last_output[-50:])
    if not egl_error:
      print("Could not record", out, "(see the error above)")
      return None
    # OSMesa only for this command: it must not leak into training.
    print("EGL rendering failed; retrying with OSMesa.")
    subprocess.run("apt-get -qq install -y libosmesa6 > /dev/null", shell=True)
    if sh("MUJOCO_GL=osmesa " + cmd, cwd=WORK, check=False) != 0:
      print("Could not record", out, "(see the error above)")
      return None
  if show:
    display(Video(str(WORK / out), embed=True, width=640))
  return WORK / out

In [ ]:
exec((WORK / "nb_helpers.py").read_text())

In [ ]:
def fetch(repo, ref, dest):
  if not (dest / ".git").exists():
    sh(f"git init -q {dest} && git -C {dest} remote add origin {repo}")
  sh(f"git -C {dest} fetch -q --depth 1 origin {ref} && git -C {dest} checkout -q FETCH_HEAD")
  sh(f"git -C {dest} log -1 --format='%h %cd %s' --date=short")

fetch("https://github.com/mujocolab/mjlab.git", MJLAB_REF, CODE / "mjlab")
fetch("https://github.com/limxdynamics/humanoid-description.git", DESCRIPTION_REF, CODE / "humanoid-description")

sh("pip install -q uv")
sh(f"uv pip install --system -q -e '{CODE / 'mjlab'}[{CUDA_EXTRA}]'")

### Write the HU_D03 package

Generated from the `hu-d03-rl` folder (`notebooks/build_kaggle_mimic.py`). You can also edit the cells here.

In [ ]:
for d in ['scripts', 'src/hu_d03_mjlab', 'src/hu_d03_mjlab/robot', 'src/hu_d03_mjlab/tasks']:
  (Path('/kaggle/temp/hu_d03_code/hu-d03-rl') / d).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/pyproject.toml
[project]
name = "hu-d03-mjlab"
version = "0.1.0"
description = "LimX HU_D03 locomotion and motion-tracking tasks for mjlab"
requires-python = ">=3.10"
dependencies = ["mjlab"]

[project.entry-points."mjlab.tasks"]
hu_d03 = "hu_d03_mjlab.tasks"

[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/__init__.py
"""LimX HU_D03 tasks for mjlab."""

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/__init__.py
"""LimX HU_D03 robot model for mjlab."""

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_spec.py
"""Load the LimX HU_D03 MJCF and turn it into a serial-joint model for mjlab.

This module only depends on `mujoco`, so it can be tested on a CPU-only
machine without mjlab installed.

What the conversion does:
  * Removes the parallel ankle (A/B achilles) and waist (A/B) linkages and the 6
    `<connect>` constraints that close them. The policy then commands the
    serial joints `ankle_pitch/roll` and `waist_roll/pitch` directly, like the
    Unitree G1 model in mjlab. The linkage mass is added to the parent body.
  * Removes the floor, light, camera, actuators and sensors from the source XML
    (mjlab adds its own terrain, actuators and sensors).
  * Sets every geom margin to 0 (MuJoCo Warp does not support margins on
    box/mesh pairs, such as the two foot boxes).
  * Names every collision geom (`<body>_collision<i>`, feet are
    `left_foot_collision` / `right_foot_collision`) so mjlab can match them.
  * Adds foot sites at the sole and the IMU sensors mjlab's tasks read.
"""

import os
from pathlib import Path

import mujoco
import numpy as np

_DEFAULT_DESCRIPTION_DIR = (
  Path(__file__).resolve().parents[3]
  / "third_party"
  / "humanoid-description"
  / "HU_D03_description"
)


def description_dir() -> Path:
  """Directory of `HU_D03_description` (override with $HU_D03_DESCRIPTION_DIR)."""
  return Path(os.environ.get("HU_D03_DESCRIPTION_DIR", _DEFAULT_DESCRIPTION_DIR))


def source_xml() -> Path:
  xml = description_dir() / "xml" / "HU_D03_03.xml"
  if not xml.exists():
    raise FileNotFoundError(
      f"{xml} not found. Clone https://github.com/limxdynamics/humanoid-description "
      "and set HU_D03_DESCRIPTION_DIR to its HU_D03_description folder."
    )
  return xml


# Bodies (with their subtrees) that form the closed-loop linkages.
LINKAGE_BODIES = (
  "left_A_achilles_link",
  "left_B_achilles_link",
  "right_A_achilles_link",
  "right_B_achilles_link",
  "waist_A_link",
  "waist_B_link",
)

# Foot sole: the foot box is centred 0.0535 m below the ankle roll frame and is
# 0.01 m half-thick, so the sole is at z = -0.0635.
FOOT_SITE_POS = (0.018, 0.0, -0.0635)
FOOT_GEOMS = {"left_foot": "left_foot_collision", "right_foot": "right_foot_collision"}


def _subtree_mass(body: mujoco.MjsBody) -> float:
  mass = body.mass
  for child in body.bodies:
    mass += _subtree_mass(child)
  return mass


def _add_mass(body: mujoco.MjsBody, extra: float) -> None:
  """Add point-like mass to a body, scaling its inertia by the same ratio."""
  ratio = (body.mass + extra) / body.mass
  body.mass = body.mass + extra
  body.inertia = np.asarray(body.inertia) * ratio


def get_spec() -> mujoco.MjSpec:
  spec = mujoco.MjSpec.from_file(str(source_xml()))

  # Equality constraints, actuators and sensors all reference linkage joints.
  for eq in list(spec.equalities):
    spec.delete(eq)
  for act in list(spec.actuators):
    spec.delete(act)
  for sensor in list(spec.sensors):
    spec.delete(sensor)

  # Floor, light and tracking camera live in the worldbody; mjlab adds terrain.
  for geom in list(spec.worldbody.geoms):
    spec.delete(geom)
  for light in list(spec.worldbody.lights):
    spec.delete(light)
  for cam in list(spec.worldbody.cameras):
    spec.delete(cam)

  # Remove the linkages, moving their mass to the parent body.
  for name in LINKAGE_BODIES:
    body = spec.body(name)
    parent = body.parent
    _add_mass(parent, _subtree_mass(body))
    spec.delete(body)

  # The free joint has a tiny spring/damper in the source XML; drop them.
  root = spec.body("base_link")
  free = root.joints[0]
  free.damping = np.zeros_like(free.damping)
  free.stiffness = np.zeros_like(free.stiffness)

  # Name collision geoms so mjlab's regex-based configs can find them.
  for body in spec.bodies:
    i = 0
    for geom in body.geoms:
      # The XML default gives every geom margin=0.001. MuJoCo Warp rejects a
      # non-zero margin on box/mesh pairs (e.g. foot-foot) with MULTICCD.
      geom.margin = 0.0
      if geom.contype == 0 and geom.conaffinity == 0:
        continue  # Visual mesh.
      if geom.name in FOOT_GEOMS:
        geom.name = FOOT_GEOMS[geom.name]
      elif not geom.name:
        geom.name = f"{body.name}_collision{i}"
        i += 1

  for side in ("left", "right"):
    spec.body(f"{side}_ankle_roll_link").add_site(
      name=f"{side}_foot", pos=FOOT_SITE_POS, size=(0.01, 0.01, 0.01), group=5
    )

  # Sensors read by mjlab's velocity and tracking tasks.
  spec.add_sensor(name="imu_ang_vel", type=mujoco.mjtSensor.mjSENS_GYRO,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_lin_vel", type=mujoco.mjtSensor.mjSENS_VELOCIMETER,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_lin_acc", type=mujoco.mjtSensor.mjSENS_ACCELEROMETER,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_upvector", type=mujoco.mjtSensor.mjSENS_FRAMEZAXIS,
                  objtype=mujoco.mjtObj.mjOBJ_BODY, objname="world",
                  reftype=mujoco.mjtObj.mjOBJ_SITE, refname="imu")
  spec.add_sensor(name="root_angmom", type=mujoco.mjtSensor.mjSENS_SUBTREEANGMOM,
                  objtype=mujoco.mjtObj.mjOBJ_BODY, objname="base_link")

  return spec

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_params.py
"""HU_D03 actuator and pose parameters (plain Python, no mjlab import).

Armatures are the reflected rotor inertias from the LimX SRDF
(`rotor mass * gear_ratio**2`), which match the `armature` values in the MJCF.
Effort limits are the `ctrlrange` of the MJCF motors.

LimX does not publish the PD gains used on the real robot, so stiffness and
damping follow mjlab's G1 rule: a second-order system with natural frequency
NATURAL_FREQ_HZ and damping ratio DAMPING_RATIO for the joint's armature.
Replace them with the real gains if you get them from LimX.

The ankle and waist roll/pitch joints are driven by two motors through a
parallel linkage. As for the G1, we assume a nominal 1:1 linkage, so these
joints get the armature and effort of both motors.
"""

import math
from dataclasses import dataclass

NATURAL_FREQ_HZ = 10.0
DAMPING_RATIO = 2.0

ARMATURE_LEG = 0.000244114 * 25.0**2  # 0.1526, hips and knees.
ARMATURE_36 = 0.000073217 * 36.0**2  # 0.0949, ankle/waist motors.
ARMATURE_ARM = 0.000073217 * 25.0**2  # 0.0458, shoulders and elbows.
ARMATURE_SMALL = 0.000017 * 25.0**2  # 0.0106, wrists, hands and head.


@dataclass(frozen=True)
class ActuatorGroup:
  joint_names_expr: tuple[str, ...]
  armature: float
  effort_limit: float

  @property
  def stiffness(self) -> float:
    w = 2.0 * math.pi * NATURAL_FREQ_HZ
    return self.armature * w**2

  @property
  def damping(self) -> float:
    w = 2.0 * math.pi * NATURAL_FREQ_HZ
    return 2.0 * DAMPING_RATIO * self.armature * w


ACTUATOR_GROUPS = {
  "leg": ActuatorGroup(
    (".*_hip_pitch_joint", ".*_hip_roll_joint", ".*_hip_yaw_joint", ".*_knee_joint"),
    ARMATURE_LEG,
    120.0,
  ),
  "ankle": ActuatorGroup(
    (".*_ankle_pitch_joint", ".*_ankle_roll_joint"), 2 * ARMATURE_36, 2 * 45.0
  ),
  "waist_yaw": ActuatorGroup(("waist_yaw_joint",), ARMATURE_36, 45.0),
  "waist": ActuatorGroup(
    ("waist_roll_joint", "waist_pitch_joint"), 2 * ARMATURE_36, 2 * 45.0
  ),
  "arm": ActuatorGroup(
    (
      ".*_shoulder_pitch_joint",
      ".*_shoulder_roll_joint",
      ".*_shoulder_yaw_joint",
      ".*_elbow_joint",
    ),
    ARMATURE_ARM,
    30.0,
  ),
  "small": ActuatorGroup(
    (".*_wrist_yaw_joint", ".*_wrist_pitch_joint", ".*_hand_yaw_joint", "head_.*"),
    ARMATURE_SMALL,
    18.0,
  ),
}

# Standing pose with slightly bent knees. Signs: negative hip pitch swings the
# thigh forward, positive knee flexes, negative elbow flexes. The base height
# puts the soles on the ground (checked by scripts/stand_test.py).
HOME_BASE_HEIGHT = 0.889
HOME_JOINT_POS = {
  ".*_hip_pitch_joint": -0.25,
  ".*_knee_joint": 0.5,
  ".*_ankle_pitch_joint": -0.25,
  "left_shoulder_roll_joint": 0.15,
  "right_shoulder_roll_joint": -0.15,
  ".*_elbow_joint": -0.3,
}

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_constants.py
"""LimX HU_D03 robot config for mjlab."""

from mjlab.actuator import BuiltinPositionActuatorCfg
from mjlab.entity import EntityArticulationInfoCfg, EntityCfg
from mjlab.utils.spec_config import CollisionCfg

from hu_d03_mjlab.robot.hu_d03_params import (
  ACTUATOR_GROUPS,
  HOME_BASE_HEIGHT,
  HOME_JOINT_POS,
)
from hu_d03_mjlab.robot.hu_d03_spec import get_spec

##
# Actuator config.
##

HU_D03_ACTUATORS = tuple(
  BuiltinPositionActuatorCfg(
    target_names_expr=g.joint_names_expr,
    stiffness=g.stiffness,
    damping=g.damping,
    effort_limit=g.effort_limit,
    armature=g.armature,
  )
  for g in ACTUATOR_GROUPS.values()
)

##
# Keyframe config.
##

HOME_KEYFRAME = EntityCfg.InitialStateCfg(
  pos=(0, 0, HOME_BASE_HEIGHT),
  joint_pos=HOME_JOINT_POS,
  joint_vel={".*": 0.0},
)

##
# Collision config.
##

FOOT_GEOMS = r"^(left|right)_foot_collision$"

# All collisions, including self collisions. Feet get condim=3, the rest condim=1.
FULL_COLLISION = CollisionCfg(
  geom_names_expr=(".*_collision.*",),
  contype=1,
  conaffinity=1,
  condim={FOOT_GEOMS: 3, ".*_collision.*": 1},
  priority={FOOT_GEOMS: 1, ".*": 0},
  friction={FOOT_GEOMS: (0.6,)},
)

##
# Final config.
##

HU_D03_ARTICULATION = EntityArticulationInfoCfg(
  actuators=HU_D03_ACTUATORS,
  soft_joint_pos_limit_factor=0.9,
)


def get_hu_d03_robot_cfg() -> EntityCfg:
  """Get a fresh HU_D03 robot configuration instance."""
  return EntityCfg(
    init_state=HOME_KEYFRAME,
    collisions=(FULL_COLLISION,),
    spec_fn=get_spec,
    articulation=HU_D03_ARTICULATION,
  )


# With the 10 Hz gains, 0.25 * effort / stiffness (the G1 rule) gives only
# 0.03-0.05 rad per unit action on the legs, because the HU_D03's reflected
# inertias are large. Use a flat 0.25 rad instead.
HU_D03_ACTION_SCALE: dict[str, float] = {".*": 0.25}

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/__init__.py
"""Registers the HU_D03 tasks with mjlab (loaded via the `mjlab.tasks` entry point)."""

from mjlab.tasks.registry import register_mjlab_task
from mjlab.tasks.tracking.rl import MotionTrackingOnPolicyRunner
from mjlab.tasks.velocity.rl import VelocityOnPolicyRunner

from .rl_cfgs import hu_d03_tracking_ppo_runner_cfg, hu_d03_velocity_ppo_runner_cfg
from .tracking_env_cfgs import hu_d03_flat_tracking_env_cfg
from .velocity_env_cfgs import hu_d03_flat_env_cfg, hu_d03_rough_env_cfg

register_mjlab_task(
  task_id="Mjlab-Velocity-Flat-LimX-HU-D03",
  env_cfg=hu_d03_flat_env_cfg(),
  play_env_cfg=hu_d03_flat_env_cfg(play=True),
  rl_cfg=hu_d03_velocity_ppo_runner_cfg(),
  runner_cls=VelocityOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Velocity-Rough-LimX-HU-D03",
  env_cfg=hu_d03_rough_env_cfg(),
  play_env_cfg=hu_d03_rough_env_cfg(play=True),
  rl_cfg=hu_d03_velocity_ppo_runner_cfg(),
  runner_cls=VelocityOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Tracking-Flat-LimX-HU-D03",
  env_cfg=hu_d03_flat_tracking_env_cfg(),
  play_env_cfg=hu_d03_flat_tracking_env_cfg(play=True),
  rl_cfg=hu_d03_tracking_ppo_runner_cfg(),
  runner_cls=MotionTrackingOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Tracking-Flat-LimX-HU-D03-No-State-Estimation",
  env_cfg=hu_d03_flat_tracking_env_cfg(has_state_estimation=False),
  play_env_cfg=hu_d03_flat_tracking_env_cfg(has_state_estimation=False, play=True),
  rl_cfg=hu_d03_tracking_ppo_runner_cfg(),
  runner_cls=MotionTrackingOnPolicyRunner,
)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/rl_cfgs.py
"""PPO runner configs for the HU_D03 tasks (G1 hyperparameters)."""

from mjlab.rl import (
  RslRlModelCfg,
  RslRlOnPolicyRunnerCfg,
  RslRlPpoAlgorithmCfg,
)


def _runner_cfg(
  experiment_name: str, entropy_coef: float, save_interval: int
) -> RslRlOnPolicyRunnerCfg:
  return RslRlOnPolicyRunnerCfg(
    actor=RslRlModelCfg(
      hidden_dims=(512, 256, 128),
      activation="elu",
      obs_normalization=True,
      distribution_cfg={
        "class_name": "GaussianDistribution",
        "init_std": 1.0,
        "std_type": "scalar",
      },
    ),
    critic=RslRlModelCfg(
      hidden_dims=(512, 256, 128),
      activation="elu",
      obs_normalization=True,
    ),
    algorithm=RslRlPpoAlgorithmCfg(
      value_loss_coef=1.0,
      use_clipped_value_loss=True,
      clip_param=0.2,
      entropy_coef=entropy_coef,
      num_learning_epochs=5,
      num_mini_batches=4,
      learning_rate=1.0e-3,
      schedule="adaptive",
      gamma=0.99,
      lam=0.95,
      desired_kl=0.01,
      max_grad_norm=1.0,
    ),
    experiment_name=experiment_name,
    save_interval=save_interval,
    num_steps_per_env=24,
    max_iterations=30_000,
  )


def hu_d03_velocity_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  return _runner_cfg("hu_d03_velocity", entropy_coef=0.01, save_interval=50)


def hu_d03_tracking_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  return _runner_cfg("hu_d03_tracking", entropy_coef=0.005, save_interval=100)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/velocity_env_cfgs.py
"""LimX HU_D03 velocity environment configurations (adapted from mjlab's G1 ones)."""

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs import mdp as envs_mdp
from mjlab.envs.mdp.actions import JointPositionActionCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.sensor import (
  ContactMatch,
  ContactSensorCfg,
  ObjRef,
  RayCastSensorCfg,
  RingPatternCfg,
  TerrainHeightSensorCfg,
)
from mjlab.tasks.velocity import mdp
from mjlab.tasks.velocity.mdp import UniformVelocityCommandCfg
from mjlab.tasks.velocity.velocity_env_cfg import make_velocity_env_cfg

from hu_d03_mjlab.robot.hu_d03_constants import (
  FOOT_GEOMS,
  HU_D03_ACTION_SCALE,
  get_hu_d03_robot_cfg,
)

PELVIS = "base_link"
TORSO = "waist_pitch_link"


def hu_d03_rough_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 rough terrain velocity configuration."""
  cfg = make_velocity_env_cfg()

  cfg.sim.mujoco.ccd_iterations = 500
  cfg.sim.contact_sensor_maxmatch = 500
  cfg.sim.nconmax = 70

  cfg.scene.entities = {"robot": get_hu_d03_robot_cfg()}

  for sensor in cfg.scene.sensors or ():
    if sensor.name == "terrain_scan":
      assert isinstance(sensor, RayCastSensorCfg)
      assert isinstance(sensor.frame, ObjRef)
      sensor.frame.name = PELVIS

  site_names = ("left_foot", "right_foot")

  for sensor in cfg.scene.sensors or ():
    if sensor.name == "foot_height_scan":
      assert isinstance(sensor, TerrainHeightSensorCfg)
      sensor.frame = tuple(
        ObjRef(type="site", name=s, entity="robot") for s in site_names
      )
      sensor.pattern = RingPatternCfg.single_ring(radius=0.03, num_samples=6)

  feet_ground_cfg = ContactSensorCfg(
    name="feet_ground_contact",
    primary=ContactMatch(
      mode="subtree",
      pattern=r"^(left_ankle_roll_link|right_ankle_roll_link)$",
      entity="robot",
    ),
    secondary=ContactMatch(mode="body", pattern="terrain"),
    fields=("found", "force"),
    reduce="netforce",
    num_slots=1,
    track_air_time=True,
  )
  self_collision_cfg = ContactSensorCfg(
    name="self_collision",
    primary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    secondary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    fields=("found", "force"),
    reduce="none",
    num_slots=1,
    history_length=4,
  )
  cfg.scene.sensors = (cfg.scene.sensors or ()) + (
    feet_ground_cfg,
    self_collision_cfg,
  )

  if cfg.scene.terrain is not None and cfg.scene.terrain.terrain_generator is not None:
    cfg.scene.terrain.terrain_generator.curriculum = True

  joint_pos_action = cfg.actions["joint_pos"]
  assert isinstance(joint_pos_action, JointPositionActionCfg)
  joint_pos_action.scale = HU_D03_ACTION_SCALE

  cfg.viewer.body_name = TORSO

  twist_cmd = cfg.commands["twist"]
  assert isinstance(twist_cmd, UniformVelocityCommandCfg)
  twist_cmd.viz.z_offset = 1.75

  cfg.events["foot_friction"].params["asset_cfg"].geom_names = FOOT_GEOMS
  cfg.events["base_com"].params["asset_cfg"].body_names = (TORSO,)

  # Same std rationale as the G1: loose knees/hip pitch, tight ankle roll and
  # waist roll/pitch, moderate arms. The HU_D03 also has a head and hand yaw.
  cfg.rewards["pose"].params["std_standing"] = {".*": 0.05}
  cfg.rewards["pose"].params["std_walking"] = {
    # Lower body.
    r".*hip_pitch.*": 0.3,
    r".*hip_roll.*": 0.15,
    r".*hip_yaw.*": 0.15,
    r".*knee.*": 0.35,
    r".*ankle_pitch.*": 0.25,
    r".*ankle_roll.*": 0.1,
    # Waist and head.
    r".*waist_yaw.*": 0.2,
    r".*waist_roll.*": 0.08,
    r".*waist_pitch.*": 0.1,
    r"head_.*": 0.1,
    # Arms.
    r".*shoulder_pitch.*": 0.15,
    r".*shoulder_roll.*": 0.15,
    r".*shoulder_yaw.*": 0.1,
    r".*elbow.*": 0.15,
    r".*wrist.*": 0.3,
    r".*hand_yaw.*": 0.3,
  }
  cfg.rewards["pose"].params["std_running"] = {
    # Lower body.
    r".*hip_pitch.*": 0.5,
    r".*hip_roll.*": 0.2,
    r".*hip_yaw.*": 0.2,
    r".*knee.*": 0.6,
    r".*ankle_pitch.*": 0.35,
    r".*ankle_roll.*": 0.15,
    # Waist and head.
    r".*waist_yaw.*": 0.3,
    r".*waist_roll.*": 0.08,
    r".*waist_pitch.*": 0.2,
    r"head_.*": 0.1,
    # Arms.
    r".*shoulder_pitch.*": 0.5,
    r".*shoulder_roll.*": 0.2,
    r".*shoulder_yaw.*": 0.15,
    r".*elbow.*": 0.35,
    r".*wrist.*": 0.3,
    r".*hand_yaw.*": 0.3,
  }

  cfg.rewards["upright"].params["asset_cfg"].body_names = (TORSO,)
  cfg.rewards["body_ang_vel"].params["asset_cfg"].body_names = (TORSO,)

  for reward_name in ["foot_clearance", "foot_slip"]:
    cfg.rewards[reward_name].params["asset_cfg"].site_names = site_names

  cfg.rewards["body_ang_vel"].weight = -0.05
  cfg.rewards["angular_momentum"].weight = -0.02
  cfg.rewards["air_time"].weight = 0.0

  cfg.rewards["self_collisions"] = RewardTermCfg(
    func=mdp.self_collision_cost,
    weight=-1.0,
    params={"sensor_name": self_collision_cfg.name, "force_threshold": 10.0},
  )

  if play:
    cfg.episode_length_s = int(1e9)

    cfg.observations["actor"].enable_corruption = False
    cfg.events.pop("push_robot", None)
    cfg.terminations.pop("out_of_terrain_bounds", None)
    cfg.curriculum = {}
    cfg.events["randomize_terrain"] = EventTermCfg(
      func=envs_mdp.randomize_terrain,
      mode="reset",
      params={},
    )

    if cfg.scene.terrain is not None:
      if cfg.scene.terrain.terrain_generator is not None:
        cfg.scene.terrain.terrain_generator.curriculum = False
        cfg.scene.terrain.terrain_generator.num_cols = 5
        cfg.scene.terrain.terrain_generator.num_rows = 5
        cfg.scene.terrain.terrain_generator.border_width = 10.0

  return cfg


def hu_d03_flat_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 flat terrain velocity configuration."""
  cfg = hu_d03_rough_env_cfg(play=play)

  cfg.sim.njmax = 300
  cfg.sim.mujoco.ccd_iterations = 50
  cfg.sim.contact_sensor_maxmatch = 64
  cfg.sim.nconmax = None

  assert cfg.scene.terrain is not None
  cfg.scene.terrain.terrain_type = "plane"
  cfg.scene.terrain.terrain_generator = None

  cfg.scene.sensors = tuple(
    s for s in (cfg.scene.sensors or ()) if s.name != "terrain_scan"
  )
  del cfg.observations["actor"].terms["height_scan"]
  del cfg.observations["critic"].terms["height_scan"]

  cfg.terminations.pop("out_of_terrain_bounds", None)
  cfg.curriculum.pop("terrain_levels", None)

  if play:
    twist_cmd = cfg.commands["twist"]
    assert isinstance(twist_cmd, UniformVelocityCommandCfg)
    twist_cmd.ranges.lin_vel_x = (-1.5, 2.0)
    twist_cmd.ranges.ang_vel_z = (-0.7, 0.7)

  return cfg

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/tracking_env_cfgs.py
"""LimX HU_D03 flat tracking (mimic) environment configurations."""

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs.mdp.actions import JointPositionActionCfg
from mjlab.managers.observation_manager import ObservationGroupCfg
from mjlab.sensor import ContactMatch, ContactSensorCfg
from mjlab.tasks.tracking.mdp import MotionCommandCfg
from mjlab.tasks.tracking.tracking_env_cfg import make_tracking_env_cfg

from hu_d03_mjlab.robot.hu_d03_constants import (
  FOOT_GEOMS,
  HU_D03_ACTION_SCALE,
  get_hu_d03_robot_cfg,
)

PELVIS = "base_link"
TORSO = "waist_pitch_link"

# Bodies whose pose is tracked (same choice of links as the G1 config).
TRACKED_BODIES = (
  PELVIS,
  "left_hip_roll_link",
  "left_knee_link",
  "left_ankle_roll_link",
  "right_hip_roll_link",
  "right_knee_link",
  "right_ankle_roll_link",
  TORSO,
  "left_shoulder_roll_link",
  "left_elbow_link",
  "left_wrist_pitch_link",
  "right_shoulder_roll_link",
  "right_elbow_link",
  "right_wrist_pitch_link",
)


def hu_d03_flat_tracking_env_cfg(
  has_state_estimation: bool = True,
  play: bool = False,
) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 flat terrain tracking configuration."""
  cfg = make_tracking_env_cfg()

  cfg.scene.entities = {"robot": get_hu_d03_robot_cfg()}

  self_collision_cfg = ContactSensorCfg(
    name="self_collision",
    primary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    secondary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    fields=("found", "force"),
    reduce="none",
    num_slots=1,
    history_length=4,
  )
  cfg.scene.sensors = (self_collision_cfg,)

  joint_pos_action = cfg.actions["joint_pos"]
  assert isinstance(joint_pos_action, JointPositionActionCfg)
  joint_pos_action.scale = HU_D03_ACTION_SCALE

  motion_cmd = cfg.commands["motion"]
  assert isinstance(motion_cmd, MotionCommandCfg)
  motion_cmd.anchor_body_name = TORSO
  motion_cmd.body_names = TRACKED_BODIES

  cfg.events["foot_friction"].params["asset_cfg"].geom_names = FOOT_GEOMS
  cfg.events["base_com"].params["asset_cfg"].body_names = (TORSO,)

  cfg.terminations["ee_body_pos"].params["body_names"] = (
    "left_ankle_roll_link",
    "right_ankle_roll_link",
    "left_wrist_pitch_link",
    "right_wrist_pitch_link",
  )

  cfg.viewer.body_name = TORSO

  if not has_state_estimation:
    new_actor_terms = {
      k: v
      for k, v in cfg.observations["actor"].terms.items()
      if k not in ["motion_anchor_pos_b", "base_lin_vel"]
    }
    cfg.observations["actor"] = ObservationGroupCfg(
      terms=new_actor_terms,
      concatenate_terms=True,
      enable_corruption=True,
    )

  if play:
    cfg.episode_length_s = int(1e9)

    cfg.observations["actor"].enable_corruption = False
    cfg.events.pop("push_robot", None)

    motion_cmd.pose_range = {}
    motion_cmd.velocity_range = {}

    motion_cmd.sampling_mode = "start"

  return cfg

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/stand_test.py
"""CPU stand test for the converted HU_D03 model (plain MuJoCo, no GPU, no mjlab).

Adds PD position actuators with the gains from hu_d03_params, puts the robot in
the home pose with its soles on the ground, and holds the pose for a few
seconds. Prints the home base height, any self-contacts at the home pose, and
how far the base drifts.

  python scripts/stand_test.py [--seconds 5] [--viewer]
"""

import argparse
import re
import sys
from pathlib import Path

import mujoco
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from hu_d03_mjlab.robot.hu_d03_params import ACTUATOR_GROUPS, HOME_JOINT_POS  # noqa: E402
from hu_d03_mjlab.robot.hu_d03_spec import get_spec  # noqa: E402


def match(patterns, name):
  return any(re.fullmatch(p, name) for p in patterns)


def build_model():
  spec = get_spec()
  spec.worldbody.add_geom(
    name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=(0, 0, 1),
  )
  spec.option.timestep = 0.005
  for joint in spec.joints:
    if joint.type != mujoco.mjtJoint.mjJNT_HINGE:
      continue
    group = next(g for g in ACTUATOR_GROUPS.values() if match(g.joint_names_expr, joint.name))
    joint.armature = group.armature
    act = spec.add_actuator(name=joint.name, target=joint.name,
                            trntype=mujoco.mjtTrn.mjTRN_JOINT)
    act.set_to_position(kp=group.stiffness, kv=group.damping)
    act.forcelimited = True
    act.forcerange = (-group.effort_limit, group.effort_limit)
  return spec.compile()


def home_qpos(model):
  qpos = model.qpos0.copy()
  for j in range(1, model.njnt):
    name = model.joint(j).name
    for pattern, value in HOME_JOINT_POS.items():
      if re.fullmatch(pattern, name):
        qpos[model.jnt_qposadr[j]] = value
  # Lift or lower the base so the lowest foot site touches z = 0.
  data = mujoco.MjData(model)
  data.qpos[:] = qpos
  data.qpos[2] = 1.0
  mujoco.mj_kinematics(model, data)
  sole = min(data.site("left_foot").xpos[2], data.site("right_foot").xpos[2])
  qpos[2] = 1.0 - sole
  return qpos


def main():
  parser = argparse.ArgumentParser()
  parser.add_argument("--seconds", type=float, default=5.0)
  parser.add_argument("--viewer", action="store_true")
  args = parser.parse_args()

  model = build_model()
  data = mujoco.MjData(model)
  qpos = home_qpos(model)
  data.qpos[:] = qpos
  data.ctrl[:] = qpos[7:]
  mujoco.mj_forward(model, data)

  print(f"home base height: {qpos[2]:.4f} m, total mass {model.body_mass.sum():.2f} kg")
  floor = model.geom("floor").id
  self_contacts = [
    (model.geom(c.geom1).name, model.geom(c.geom2).name)
    for c in data.contact[: data.ncon]
    if floor not in (c.geom1, c.geom2)
  ]
  print(f"self-contacts at home pose: {self_contacts or 'none'}")

  def step():
    mujoco.mj_step(model, data)

  if args.viewer:
    from mujoco import viewer as mj_viewer

    with mj_viewer.launch_passive(model, data) as v:
      while v.is_running() and data.time < args.seconds:
        step()
        v.sync()
  else:
    while data.time < args.seconds:
      step()

  drift = np.linalg.norm(data.qpos[:2] - qpos[:2])
  up = data.xmat[model.body("base_link").id].reshape(3, 3)[:, 2]
  tilt = np.degrees(np.arccos(np.clip(up[2], -1, 1)))
  joint_err = np.abs(data.qpos[7:] - qpos[7:]).max()
  print(f"after {args.seconds:.1f}s: base z {data.qpos[2]:.4f} m, xy drift {drift:.4f} m, "
        f"tilt {tilt:.2f} deg, max joint error {joint_err:.4f} rad")
  ok = data.qpos[2] > qpos[2] - 0.1 and tilt < 10
  print("STAND TEST", "PASSED" if ok else "FAILED")
  return 0 if ok else 1


if __name__ == "__main__":
  sys.exit(main())

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/record_video.py
"""Roll out a policy headless and save an .mp4 (for Colab/Kaggle, where mjlab's
`play` viewer cannot open a window).

  python scripts/record_video.py Mjlab-Velocity-Flat-LimX-HU-D03 \
      --checkpoint-file logs/.../model_5000.pt --output-file walk.mp4
  python scripts/record_video.py Mjlab-Tracking-Flat-LimX-HU-D03 \
      --checkpoint-file ... --motion-file motion.npz --output-file mimic.mp4
  # --agent zero holds the default pose (stand test, no checkpoint needed).
  # --start-seconds 42 starts a tracking video at that time of the motion (e.g. one dance of a medley).
  # Defaults to 1280x720; e.g. --width 1920 --height 1080 --distance 4 for 1080p.
"""

from dataclasses import asdict
from pathlib import Path
from typing import Literal

import mediapy
import torch
import tyro

import mjlab
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.tasks.tracking.mdp import MotionCommandCfg


def main(
  task_id: tyro.conf.Positional[str],
  output_file: str,
  checkpoint_file: str | None = None,
  motion_file: str | None = None,
  agent: Literal["trained", "zero"] = "trained",
  seconds: float = 10.0,
  start_seconds: float = 0.0,
  no_terminations: bool = False,
  width: int = 1280,
  height: int = 720,
  distance: float = 3.0,
  elevation: float = -15.0,
  azimuth: float = 90.0,
):
  device = "cuda:0" if torch.cuda.is_available() else "cpu"
  env_cfg = load_env_cfg(task_id, play=True)
  agent_cfg = load_rl_cfg(task_id)
  env_cfg.scene.num_envs = 1
  # mjlab renders 320x240 from 5 m away by default; frame the whole robot instead.
  env_cfg.viewer.width, env_cfg.viewer.height = width, height
  env_cfg.viewer.distance = distance
  env_cfg.viewer.elevation = elevation
  env_cfg.viewer.azimuth = azimuth
  if no_terminations:
    env_cfg.terminations = {}

  motion_cmd = env_cfg.commands.get("motion")
  if isinstance(motion_cmd, MotionCommandCfg):
    if motion_file is None:
      raise ValueError("Tracking tasks need --motion-file.")
    motion_cmd.motion_file = motion_file

  env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode="rgb_array")
  env = RslRlVecEnvWrapper(env, clip_actions=agent_cfg.clip_actions)

  if agent == "zero":
    shape = env.unwrapped.action_space.shape

    def policy(obs):
      del obs
      return torch.zeros(shape, device=env.unwrapped.device)
  else:
    if checkpoint_file is None:
      raise ValueError("--checkpoint-file is required with --agent trained.")
    runner_cls = load_runner_cls(task_id) or MjlabOnPolicyRunner
    runner = runner_cls(env, asdict(agent_cfg), device=device)
    runner.load(checkpoint_file, load_cfg={"actor": True}, strict=True,
                map_location=device)
    policy = runner.get_inference_policy(device=device)

  if start_seconds > 0:
    # Put the robot exactly in the reference state at that frame (play mode starts at frame 0).
    command = env.unwrapped.command_manager.get_term("motion")
    frame = round(start_seconds / env.unwrapped.step_dt)
    command.reset_to_frame(torch.arange(env.num_envs, device=env.unwrapped.device), frame)
    env.unwrapped.sim.forward()
    command.update_relative_body_poses()

  obs = env.get_observations()
  frames = []
  steps = int(seconds / env.unwrapped.step_dt)
  with torch.inference_mode():
    for _ in range(steps):
      obs, _, _, _ = env.step(policy(obs))
      frames.append(env.unwrapped.render())

  fps = round(1.0 / env.unwrapped.step_dt)
  Path(output_file).parent.mkdir(parents=True, exist_ok=True)
  mediapy.write_video(output_file, frames, fps=fps)
  print(f"wrote {output_file} ({len(frames)} frames at {fps} fps)")
  env.close()


if __name__ == "__main__":
  tyro.cli(main, config=mjlab.TYRO_FLAGS)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/retarget_g1_to_hu_d03.py
"""Convert a Unitree G1 motion CSV (LAFAN1 retargeting dataset format) to the HU_D03.

Joint-space transfer, plain MuJoCo on CPU:
  * Joints with the same name are copied. The sign of each joint is chosen by
    comparing the two robots' joint axes in the world frame at the zero pose.
  * The G1 wrist pitch/yaw go to the HU_D03 wrist pitch / hand yaw; the G1
    wrist roll and the HU_D03 head stay at 0.
  * The elbows get a fixed offset because the zero poses differ (ZERO_OFFSETS).
  * Values are clipped to the HU_D03 joint limits.
  * The root height is scaled by the ratio of pelvis heights, then each frame
    is shifted so the feet do not go through the ground.

The HU_D03 is ~1.6 m tall and the G1 ~1.3 m, so foot placement and hand
positions are only approximate. It is good enough to train a tracking policy;
for exact contacts, retarget from the human data with an IK tool instead.

Input CSV (30 fps): x y z qx qy qz qw + 29 G1 joints.
Output CSV (same fps): x y z qx qy qz qw + 31 HU_D03 joints (model order).

  python scripts/retarget_g1_to_hu_d03.py --input walk1_subject1.csv \
      --output walk1_subject1_hu_d03.csv [--g1-xml path/to/g1.xml]
"""

import argparse
import sys
from pathlib import Path

import mujoco
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from hu_d03_mjlab.robot.hu_d03_spec import get_spec  # noqa: E402

G1_JOINTS = (
  "left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint",
  "left_knee_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint",
  "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",
  "right_knee_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",
  "waist_yaw_joint", "waist_roll_joint", "waist_pitch_joint",
  "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint",
  "left_elbow_joint", "left_wrist_roll_joint", "left_wrist_pitch_joint",
  "left_wrist_yaw_joint",
  "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint",
  "right_elbow_joint", "right_wrist_roll_joint", "right_wrist_pitch_joint",
  "right_wrist_yaw_joint",
)  # fmt: skip

# HU_D03 joint -> G1 joint, where the names differ.
RENAMED = {
  "left_hand_yaw_joint": "left_wrist_yaw_joint",
  "right_hand_yaw_joint": "right_wrist_yaw_joint",
}
# Zero-pose differences (HU_D03 angle = sign * G1 angle + offset). At zero the
# G1 forearm points forward, the HU_D03 forearm hangs down; found by matching
# the forearm direction of both robots at the G1 zero pose.
ZERO_OFFSETS = {"left_elbow_joint": -1.4984, "right_elbow_joint": -1.4984}
UNMAPPED = {"head_yaw_joint", "head_pitch_joint", "left_wrist_yaw_joint", "right_wrist_yaw_joint"}

HU_FOOT_POINTS = tuple(
  f"contact_{side}_{p}" for side in ("left", "right") for p in ("heel", "center", "tip")
)


def default_g1_xml() -> Path:
  try:
    from mjlab import MJLAB_SRC_PATH

    return MJLAB_SRC_PATH / "asset_zoo/robots/unitree_g1/xmls/g1.xml"
  except ImportError:
    root = Path(__file__).resolve().parents[1]
    return root / "third_party/mjlab/src/mjlab/asset_zoo/robots/unitree_g1/xmls/g1.xml"


def world_axes(model: mujoco.MjModel) -> dict[str, np.ndarray]:
  data = mujoco.MjData(model)
  mujoco.mj_kinematics(model, data)
  mujoco.mj_comPos(model, data)
  return {
    model.joint(j).name: data.xaxis[j].copy()
    for j in range(model.njnt)
    if model.jnt_type[j] == mujoco.mjtJoint.mjJNT_HINGE
  }


def pelvis_height(model: mujoco.MjModel, foot_sites: tuple[str, ...]) -> float:
  data = mujoco.MjData(model)
  data.qpos[2] = 1.0
  mujoco.mj_kinematics(model, data)
  return 1.0 - min(data.site(s).xpos[2] for s in foot_sites)


def build_mapping(g1: mujoco.MjModel, hu: mujoco.MjModel):
  """Returns (hu_qpos_index, g1_column, sign) triples and a printable report."""
  g1_axes, hu_axes = world_axes(g1), world_axes(hu)
  mapping, report = [], []
  for j in range(1, hu.njnt):
    name = hu.joint(j).name
    if name in UNMAPPED:
      report.append(f"  {name:28s} <- 0")
      continue
    src = RENAMED.get(name, name)
    dot = float(np.dot(g1_axes[src], hu_axes[name]))
    if abs(dot) < 0.7:
      report.append(f"  {name:28s} <- 0 (axes differ, dot={dot:+.2f})")
      continue
    sign = 1.0 if dot > 0 else -1.0
    mapping.append((hu.jnt_qposadr[j], G1_JOINTS.index(src), sign))
    report.append(f"  {name:28s} <- {'+' if sign > 0 else '-'}{src} (dot={dot:+.2f})")
  return mapping, report


def retarget(motion: np.ndarray, g1: mujoco.MjModel, hu: mujoco.MjModel, verbose=True):
  mapping, report = build_mapping(g1, hu)
  if verbose:
    print("joint mapping:\n" + "\n".join(report))

  scale = pelvis_height(hu, ("left_foot", "right_foot")) / pelvis_height(
    g1, ("left_foot", "right_foot")
  )
  lo = hu.jnt_range[1:, 0] * 0.98
  hi = hu.jnt_range[1:, 1] * 0.98

  n = motion.shape[0]
  qpos = np.zeros((n, hu.nq))
  qpos[:, :3] = motion[:, :3] * scale
  qpos[:, 3:7] = motion[:, [6, 3, 4, 5]]  # xyzw -> wxyz
  for adr, col, sign in mapping:
    qpos[:, adr] = sign * motion[:, 7 + col]
  for name, offset in ZERO_OFFSETS.items():
    qpos[:, hu.jnt_qposadr[hu.joint(name).id]] += offset
  qpos[:, 7:] = np.clip(qpos[:, 7:], lo, hi)

  # Ground the feet: global shift so the feet touch the ground in most frames,
  # then lift any frame where a foot point is still below the ground.
  data = mujoco.MjData(hu)
  min_z = np.empty(n)
  for i in range(n):
    data.qpos[:] = qpos[i]
    mujoco.mj_kinematics(hu, data)
    min_z[i] = min(data.body(b).xpos[2] for b in HU_FOOT_POINTS)
  qpos[:, 2] -= np.percentile(min_z, 5)
  min_z -= np.percentile(min_z, 5)
  qpos[:, 2] += np.maximum(-min_z, 0.0)

  if verbose:
    print(f"root height scale {scale:.3f}, frames {n}")
  out = np.concatenate([qpos[:, :3], qpos[:, [4, 5, 6, 3]], qpos[:, 7:]], axis=1)
  return out


def main():
  parser = argparse.ArgumentParser(description=__doc__.split("\n")[0])
  parser.add_argument("--input", required=True)
  parser.add_argument("--output", required=True)
  parser.add_argument("--g1-xml", default=None)
  args = parser.parse_args()

  g1 = mujoco.MjModel.from_xml_path(str(args.g1_xml or default_g1_xml()))
  hu = get_spec().compile()
  motion = np.loadtxt(args.input, delimiter=",")
  assert motion.shape[1] == 7 + len(G1_JOINTS), motion.shape
  out = retarget(motion, g1, hu)
  np.savetxt(args.output, out, delimiter=",", fmt="%.6f")
  print(f"wrote {args.output} ({out.shape[0]} frames, {out.shape[1] - 7} joints)")


if __name__ == "__main__":
  main()

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/csv_to_npz_hu_d03.py
"""Convert an HU_D03 motion CSV to the .npz that mjlab's tracking task reads.

Same as mjlab's scripts/csv_to_npz.py (which hard-codes the G1), but for the
HU_D03, and it writes the .npz to a local path instead of requiring a Weights &
Biases registry upload. Train with:

  train Mjlab-Tracking-Flat-LimX-HU-D03 --env.commands.motion.motion-file motion.npz

  python scripts/csv_to_npz_hu_d03.py --input-file walk_hu_d03.csv \
      --output-file walk_hu_d03.npz [--input-fps 30] [--output-fps 50]
"""

from pathlib import Path

import numpy as np
import torch
import tyro
from tqdm import tqdm

import mjlab  # noqa: F401  (configures Warp, loads the HU_D03 tasks)
from mjlab.entity import Entity
from mjlab.scene import Scene
from mjlab.scripts.csv_to_npz import MotionLoader
from mjlab.sim.sim import Simulation, SimulationCfg

from hu_d03_mjlab.tasks.tracking_env_cfgs import hu_d03_flat_tracking_env_cfg

LOG_KEYS = (
  "joint_pos",
  "joint_vel",
  "body_pos_w",
  "body_quat_w",
  "body_lin_vel_w",
  "body_ang_vel_w",
)


def main(
  input_file: str,
  output_file: str,
  input_fps: float = 30.0,
  output_fps: float = 50.0,
  device: str = "cuda:0",
  line_range: tuple[int, int] | None = None,
):
  """Args:
    line_range: Only convert these CSV lines (1-based, inclusive), e.g. (1, 1500).
  """
  if device.startswith("cuda") and not torch.cuda.is_available():
    print("[WARNING]: CUDA is not available. Falling back to CPU.")
    device = "cpu"

  sim_cfg = SimulationCfg()
  sim_cfg.mujoco.timestep = 1.0 / output_fps

  scene = Scene(hu_d03_flat_tracking_env_cfg().scene, device=device)
  model = scene.compile()
  sim = Simulation(num_envs=1, cfg=sim_cfg, model=model, device=device)
  scene.initialize(sim.mj_model, sim.model, sim.data)

  robot: Entity = scene["robot"]
  # CSV columns are the HU_D03 joints in model order.
  joint_names = list(robot.joint_names)

  motion = MotionLoader(
    motion_file=input_file,
    input_fps=int(input_fps),
    output_fps=int(output_fps),
    device=sim.device,
    line_range=line_range,
  )
  assert motion.motion_dof_poss_input.shape[1] == len(joint_names), (
    f"CSV has {motion.motion_dof_poss_input.shape[1]} joints, robot has "
    f"{len(joint_names)}"
  )
  joint_ids = robot.find_joints(joint_names, preserve_order=True)[0]

  log: dict[str, list] = {k: [] for k in LOG_KEYS}
  scene.reset()

  for _ in tqdm(range(motion.output_frames), desc="frames"):
    (
      (base_pos, base_rot, base_lin_vel, base_ang_vel, dof_pos, dof_vel),
      _,
    ) = motion.get_next_state()

    root_states = robot.data.default_root_state.clone()
    root_states[:, 0:3] = base_pos
    root_states[:, :2] += scene.env_origins[:, :2]
    root_states[:, 3:7] = base_rot
    root_states[:, 7:10] = base_lin_vel
    root_states[:, 10:] = base_ang_vel
    robot.write_root_state_to_sim(root_states)

    joint_pos = robot.data.default_joint_pos.clone()
    joint_vel = robot.data.default_joint_vel.clone()
    joint_pos[:, joint_ids] = dof_pos
    joint_vel[:, joint_ids] = dof_vel
    robot.write_joint_state_to_sim(joint_pos, joint_vel)

    sim.forward()
    scene.update(sim.mj_model.opt.timestep)

    data = robot.data
    log["joint_pos"].append(data.joint_pos[0].cpu().numpy().copy())
    log["joint_vel"].append(data.joint_vel[0].cpu().numpy().copy())
    log["body_pos_w"].append(data.body_link_pos_w[0].cpu().numpy().copy())
    log["body_quat_w"].append(data.body_link_quat_w[0].cpu().numpy().copy())
    log["body_lin_vel_w"].append(data.body_link_lin_vel_w[0].cpu().numpy().copy())
    log["body_ang_vel_w"].append(data.body_link_ang_vel_w[0].cpu().numpy().copy())

  out = {"fps": [output_fps], **{k: np.stack(v, axis=0) for k, v in log.items()}}
  Path(output_file).parent.mkdir(parents=True, exist_ok=True)
  np.savez(output_file, **out)
  print(f"wrote {output_file} ({len(log['joint_pos'])} frames at {output_fps} fps)")


if __name__ == "__main__":
  tyro.cli(main, config=mjlab.TYRO_FLAGS)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/seed_g1_to_lafan_csv.py
"""Convert a BONES-SEED G1 CSV to the LAFAN1 G1 CSV format retarget_g1_to_hu_d03.py reads.

BONES-SEED (huggingface.co/datasets/bones-studio/seed, g1/csv/<date>/<name>.csv), 120 fps:
  header row; Frame, root_translateX/Y/Z (cm), root_rotateX/Y/Z (extrinsic XYZ
  Euler, degrees), <joint>_dof (degrees, G1 MJCF joint names).
LAFAN1 G1 format, 30 fps by default:
  no header; x y z (m), qx qy qz qw, 29 joints (radians) in G1_JOINTS order.

  python scripts/seed_g1_to_lafan_csv.py --input ballet.csv --output ballet_g1.csv \
      [--input-fps 120] [--output-fps 30] [--start-s 0] [--end-s 30]
"""

import argparse
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))

from retarget_g1_to_hu_d03 import G1_JOINTS  # noqa: E402


def euler_xyz_extrinsic_to_quat_xyzw(euler_deg: np.ndarray) -> np.ndarray:
  """Extrinsic X, then Y, then Z rotations (R = Rz Ry Rx) as xyzw quaternions."""
  half = np.deg2rad(euler_deg) / 2
  cx, cy, cz = np.cos(half).T
  sx, sy, sz = np.sin(half).T
  # q = qz * qy * qx
  w = cz * cy * cx + sz * sy * sx
  x = cz * cy * sx - sz * sy * cx
  y = cz * sy * cx + sz * cy * sx
  z = sz * cy * cx - cz * sy * sx
  return np.stack([x, y, z, w], axis=1)


def convert(path: str, input_fps: int, output_fps: int, start_s: float, end_s: float | None):
  with open(path) as f:
    header = [h.strip() for h in f.readline().split(",")]
  data = np.loadtxt(path, delimiter=",", skiprows=1, ndmin=2)
  col = {name: i for i, name in enumerate(header)}

  pos = data[:, [col[f"root_translate{a}"] for a in "XYZ"]] / 100.0
  quat = euler_xyz_extrinsic_to_quat_xyzw(data[:, [col[f"root_rotate{a}"] for a in "XYZ"]])

  # Match joints by name; fall back to column order if the names differ.
  names = [f"{j}_dof" for j in G1_JOINTS]
  if all(n in col for n in names):
    joints = data[:, [col[n] for n in names]]
  else:
    joints = data[:, 7:]
    assert joints.shape[1] == len(G1_JOINTS), (
      f"expected {len(G1_JOINTS)} joint columns, got {joints.shape[1]}: {header[7:]}"
    )
    print("joint names differ from the G1 MJCF; using column order:", header[7:])
  joints = np.deg2rad(joints)

  if input_fps % output_fps:
    raise ValueError(f"input fps {input_fps} is not a multiple of output fps {output_fps}")
  first = int(round(start_s * input_fps))
  last = None if end_s is None else int(round(end_s * input_fps))
  step = input_fps // output_fps
  return np.concatenate([pos, quat, joints], axis=1)[first:last:step]


def main():
  parser = argparse.ArgumentParser(description=__doc__.split("\n")[0])
  parser.add_argument("--input", required=True)
  parser.add_argument("--output", required=True)
  parser.add_argument("--input-fps", type=int, default=120)
  parser.add_argument("--output-fps", type=int, default=30)
  parser.add_argument("--start-s", type=float, default=0.0)
  parser.add_argument("--end-s", type=float, default=None)
  args = parser.parse_args()

  out = convert(args.input, args.input_fps, args.output_fps, args.start_s, args.end_s)
  np.savetxt(args.output, out, delimiter=",", fmt="%.6f")
  print(f"wrote {args.output} ({out.shape[0]} frames at {args.output_fps} fps, "
        f"{out.shape[0] / args.output_fps:.1f} s)")


if __name__ == "__main__":
  main()

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/make_medley_hu_d03.py
"""Join several HU_D03 motion CSVs into one continuous clip (a "medley").

One tracking policy trained on the medley learns all of its dances: the policy
observes the reference joint positions and velocities, so it tracks whichever
part of the clip it is given, and mjlab's adaptive sampling spends more time on
the parts it fails at.

Each clip is turned about the vertical axis and shifted so it starts where the
previous one ended (same heading and floor position), then a short blend is
inserted: joints and root height are interpolated with a smoothstep, the root
orientation is slerped. Plain numpy, CPU.

Input/output CSV (30 fps): x y z qx qy qz qw + HU_D03 joints (model order).
A <output>.segments.json next to the output lists where each clip starts and ends.

  python scripts/make_medley_hu_d03.py --inputs a_hu_d03.csv b_hu_d03.csv \
      --output medley_hu_d03.csv [--blend-seconds 1.0] [--fps 30]
"""

import argparse
import json
from pathlib import Path

import numpy as np


def quat_mul(a, b):
  """Hamilton product of xyzw quaternions (broadcasts)."""
  ax, ay, az, aw = np.moveaxis(a, -1, 0)
  bx, by, bz, bw = np.moveaxis(b, -1, 0)
  return np.stack([
    aw * bx + ax * bw + ay * bz - az * by,
    aw * by - ax * bz + ay * bw + az * bx,
    aw * bz + ax * by - ay * bx + az * bw,
    aw * bw - ax * bx - ay * by - az * bz,
  ], axis=-1)


def yaw(q):
  x, y, z, w = q
  return np.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))


def slerp(q0, q1, t):
  d = np.dot(q0, q1)
  if d < 0:
    q1, d = -q1, -d
  if d > 0.9995:
    q = q0 + t * (q1 - q0)
    return q / np.linalg.norm(q)
  th = np.arccos(d)
  return (np.sin((1 - t) * th) * q0 + np.sin(t * th) * q1) / np.sin(th)


def align(clip, prev):
  """Turn and shift clip so its first frame has prev's heading and floor position."""
  dyaw = yaw(prev[3:7]) - yaw(clip[0, 3:7])
  c, s = np.cos(dyaw), np.sin(dyaw)
  out = clip.copy()
  xy = clip[:, :2] - clip[0, :2]
  out[:, 0] = c * xy[:, 0] - s * xy[:, 1] + prev[0]
  out[:, 1] = s * xy[:, 0] + c * xy[:, 1] + prev[1]
  qz = np.array([0.0, 0.0, np.sin(dyaw / 2), np.cos(dyaw / 2)])
  out[:, 3:7] = quat_mul(qz, clip[:, 3:7])
  return out


def blend(a, b, n):
  """n frames going from frame a to frame b (both excluded)."""
  frames = []
  for k in range(1, n + 1):
    t = k / (n + 1)
    t = t * t * (3 - 2 * t)  # Smoothstep: zero velocity change at both ends of the blend.
    f = a + t * (b - a)
    f[3:7] = slerp(a[3:7], b[3:7], t)
    frames.append(f)
  return np.array(frames).reshape(n, a.shape[0])


def main():
  parser = argparse.ArgumentParser(description=__doc__.split("\n")[0])
  parser.add_argument("--inputs", nargs="+", required=True)
  parser.add_argument("--output", required=True)
  parser.add_argument("--blend-seconds", type=float, default=1.0)
  parser.add_argument("--fps", type=float, default=30.0)
  args = parser.parse_args()

  n_blend = round(args.blend_seconds * args.fps)
  out = np.loadtxt(args.inputs[0], delimiter=",", ndmin=2)
  segments = [{"clip": Path(args.inputs[0]).stem, "start_s": 0.0, "end_s": len(out) / args.fps}]
  for path in args.inputs[1:]:
    clip = np.loadtxt(path, delimiter=",", ndmin=2)
    assert clip.shape[1] == out.shape[1], (path, clip.shape)
    clip = align(clip, out[-1])
    out = np.concatenate([out, blend(out[-1], clip[0], n_blend), clip])
    segments.append({"clip": Path(path).stem, "start_s": (len(out) - len(clip)) / args.fps,
                     "end_s": len(out) / args.fps})

  np.savetxt(args.output, out, delimiter=",", fmt="%.6f")
  Path(args.output).with_suffix(".segments.json").write_text(json.dumps(segments, indent=1))
  for s in segments:
    print(f"  {s['start_s']:7.1f} - {s['end_s']:7.1f} s  {s['clip']}")
  print(f"wrote {args.output} ({len(out)} frames, {len(out) / args.fps:.1f} s, "
        f"{len(segments)} clips, {args.blend_seconds:g} s blends)")


if __name__ == "__main__":
  main()

In [ ]:
sh(f"uv pip install --system -q -e {CODE / 'hu-d03-rl'}")

## 4. Sanity checks

* The stand test holds the home pose in plain MuJoCo (CPU).
* `list-envs` must show the `LimX-HU-D03` tasks. If it does not, look for a
  `[WARN] Failed to load task package 'hu_d03'` traceback.
* Video problems never stop the run. If rendering fails with an EGL error, the video helper switches to
  OSMesa for that video and tries again.

In [ ]:
SCRIPTS = CODE / "hu-d03-rl" / "scripts"
sh(f"python {SCRIPTS / 'stand_test.py'} --seconds 10")
sh("list-envs | grep -i hu-d03")

## 5. Prepare the motion

Gets the G1 clip (from `/kaggle/input` if a Kaggle dataset or earlier version there has it), converts a
BONES-SEED clip to the LAFAN1 layout (`seed_g1_to_lafan_csv.py`: cm → m, Euler → quaternion, 120 → 30 fps),
copies it to the HU_D03 joints (`retarget_g1_to_hu_d03.py`: joint signs from the joint axes, elbow zero-pose
offset, feet kept above the ground), then computes the body poses and velocities mjlab's tracking task needs
(`csv_to_npz_hu_d03.py`, on the GPU). The `.npz` is saved in the output next to the logs.

BONES-SEED ships all G1 clips in one 23.5 GB archive. It is streamed once and only the clips you picked are kept,
which takes roughly 5–30 minutes the first time; the extracted CSVs are saved in the output, so later versions
skip it. For a medley, the retargeted clips are then joined (`make_medley_hu_d03.py`, see *Several dances in one
policy* at the top) and `<clip>.segments.json` lists where each dance starts.

In [ ]:
import shlex
data = WORK / "motions"
data.mkdir(exist_ok=True)
safe = lambda name: re.sub(r"[^A-Za-z0-9_.-]+", "_", name)

if SOURCE == "seed":
  missing = []
  for name in MOTIONS:
    seed_csv = data / f"{safe(name)}_seed.csv"
    local = sorted(Path("/kaggle/input").glob(f"**/{safe(name)}_seed.csv"))
    if not seed_csv.exists() and local:
      shutil.copy(local[0], seed_csv)
      print("using", local[0])
    if not seed_csv.exists():
      missing.append(name)
  if missing:
    # Stream the archive once and stop when every member matching <date>/<name>.csv has been found.
    members = [f"*{Path(SEED_PATHS[n]).parent.name}/{n}.csv" if SEED_PATHS[n] else f"*/{n}.csv" for n in missing]
    tmp = data / "seed_extract"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir()
    url = "https://huggingface.co/datasets/bones-studio/seed/resolve/main/g1.tar.gz"
    print(f"streaming g1.tar.gz (23.5 GB) to extract {len(members)} clip(s) ...")
    sh(f'curl -sSfL -H "Authorization: Bearer $HF_TOKEN" {url} '
       f'| tar -xzf - -C {tmp} --wildcards --occurrence=1 {" ".join(map(shlex.quote, members))}', check=False)
    for name in missing:
      found = [p for p in tmp.rglob("*.csv") if p.name == f"{name}.csv"]
      if not found:
        raise SystemExit(f"{name}.csv not found in g1.tar.gz (path in metadata: {SEED_PATHS[name]}). "
                         "Check the HF_TOKEN secret and that your access request was accepted.")
      shutil.move(found[0], data / f"{safe(name)}_seed.csv")
    shutil.rmtree(tmp)

hu_csvs = []
for name in MOTIONS:
  g1_csv = data / f"{safe(name)}_g1.csv"
  if SOURCE == "seed":
    seed_csv = data / f"{safe(name)}_seed.csv"
    sh(f"python {SCRIPTS / 'seed_g1_to_lafan_csv.py'} --input {shlex.quote(str(seed_csv))} --output {g1_csv}")
  elif not g1_csv.exists():
    local = sorted(Path("/kaggle/input").glob(f"**/{name}.csv"))
    if local:
      shutil.copy(local[0], g1_csv)
      print("using", local[0])
    else:
      url = f"https://huggingface.co/datasets/lvhaidong/LAFAN1_Retargeting_Dataset/resolve/main/g1/{name}.csv"
      sh(f"wget -q -O {g1_csv}.part {url} && mv {g1_csv}.part {g1_csv}")
  hu_csvs.append(data / f"{safe(name)}_hu_d03.csv")
  sh(f"python {SCRIPTS / 'retarget_g1_to_hu_d03.py'} --input {g1_csv} --output {hu_csvs[-1]} "
     f"--g1-xml {CODE / 'mjlab/src/mjlab/asset_zoo/robots/unitree_g1/xmls/g1.xml'}")

if MEDLEY:
  hu_csv = data / f"{CLIP}_hu_d03.csv"
  sh(f"python {SCRIPTS / 'make_medley_hu_d03.py'} --inputs {' '.join(map(str, hu_csvs))} "
     f"--output {hu_csv} --blend-seconds {BLEND_SECONDS}")
else:
  hu_csv = hu_csvs[0]

MOTION_FILE = data / f"{RUN_NAME}_hu_d03.npz"
lr = f'--line-range "({LINE_RANGE[0]}, {LINE_RANGE[1]})"' if LINE_RANGE else ""
sh(f"python {SCRIPTS / 'csv_to_npz_hu_d03.py'} --input-file {hu_csv} --output-file {MOTION_FILE} {lr}")
print("motion file:", MOTION_FILE)

Replay the reference (zero actions, terminations off) to check the retargeting. If the motion looks wrong here,
the policy cannot learn it: fix the clip before training.

In [ ]:
video(f"videos/{CLIP}_reference.mp4", "--agent zero", "--no-terminations True", "--seconds 10")

## 6. Train

Logs, checkpoints (every 100 iterations) and ONNX exports go to `/kaggle/working/<OUTPUT_NAME>/logs/rsl_rl/hu_d03_tracking/<date_time>/`.
Training is stopped after `TIME_BUDGET_HOURS`; the newest checkpoint is kept.
Set `TRAIN = False` in the settings to skip this cell and only make the curves, video and zip of an existing run.

The cell shows one progress line, updated every 10 s. The percentage is of whichever limit comes first,
`MAX_ITERATIONS` or `TIME_BUDGET_HOURS`. The full rsl_rl output goes to `<OUTPUT_NAME>/train.log`
(`!tail -n 60 /kaggle/working/<OUTPUT_NAME>/train.log` shows the latest iteration in detail).

In [ ]:
cmd = (f"MUJOCO_GL=egl train {TASK} --env.scene.num-envs {NUM_ENVS} "
       f"--agent.max-iterations {MAX_ITERATIONS} --agent.logger tensorboard "
       f"--env.commands.motion.motion-file {MOTION_FILE} --agent.run-name {RUN_NAME}")
if USE_TWO_GPUS and NUM_GPUS >= 2:
  cmd += ' --gpu-ids "[0, 1]"'
if RESUME_RUN is not None:
  cmd += f" --agent.resume True --agent.load-run {RESUME_RUN.name}"
elif INIT_RUN is not None:
  cmd += f" --agent.resume True --agent.load-run {INIT_RUN} --agent.load-checkpoint {INIT_CHECKPOINT}"
  # Warm start: keep the weights but explore again and count iterations from 0 (edits the copy in LOG_ROOT).
  import torch
  ckpt_path = LOG_ROOT / INIT_RUN / INIT_CHECKPOINT
  ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
  actor = ckpt["actor_state_dict"]
  if WARM_START_STD and "distribution.std_param" in actor:
    print(f"action std {actor['distribution.std_param'].mean():.2f} -> at least {WARM_START_STD}")
    actor["distribution.std_param"].clamp_(min=WARM_START_STD)
  for group in ckpt["optimizer_state_dict"]["param_groups"]:
    group["lr"] = 1.0e-3
  ckpt["iter"] = 0
  torch.save(ckpt, ckpt_path)
import html, re
from datetime import timedelta
import IPython.display as ipd

class TrainProgress:
  """Parses rsl_rl's per-iteration log and shows one progress line."""
  ANSI = re.compile(r"\x1b\[[0-9;]*m")
  FIELDS = {
    "iteration": re.compile(r"Learning iteration (\d+)/(\d+)"),
    "reward": re.compile(r"Mean reward:\s+(-?[\d.]+)"),
    "length": re.compile(r"Mean episode length:\s+([\d.]+)"),
    "fps": re.compile(r"Steps per second:\s+([\d.]+)"),
  }

  def __init__(self, budget_s):
    self.budget_s = budget_s
    self.values = {}
    self.handle = ipd.display(ipd.HTML("<pre>starting training...</pre>"), display_id=True)

  def on_line(self, line):
    line = self.ANSI.sub("", line)
    for key, pattern in self.FIELDS.items():
      m = pattern.search(line)
      if m:
        self.values[key] = m.groups()

  def on_tick(self, elapsed):
    v = self.values
    if "iteration" not in v:
      text = f"setting up (compiling GPU kernels)...  {timedelta(seconds=int(elapsed))}"
    else:
      it, total = (int(x) for x in v["iteration"])
      frac = min(1.0, max((it + 1) / total, elapsed / self.budget_s))
      bar = "█" * int(frac * 30) + "░" * (30 - int(frac * 30))
      text = f"{bar} {frac * 100:5.1f}%  iter {it + 1}/{total}"
      if "reward" in v:
        text += f"  reward {float(v['reward'][0]):.2f}  ep len {float(v['length'][0]):.0f}/{EPISODE_STEPS}"
      if "fps" in v:
        text += f"  {float(v['fps'][0]):,.0f} steps/s"
      text += f"  {timedelta(seconds=int(elapsed))} / {timedelta(seconds=int(self.budget_s))}"
    self.handle.update(ipd.HTML(f"<pre>{html.escape(text)}</pre>"))

if TRAIN:
  progress = TrainProgress(TIME_BUDGET_HOURS * 3600)
  sh(cmd, cwd=WORK, max_seconds=TIME_BUDGET_HOURS * 3600, log_file=WORK / "train.log",
     on_line=progress.on_line, on_tick=progress.on_tick)
else:
  print("TRAIN = False: skipping training")

### Restore after a restart

If the notebook restarted after training (idle timeout, crash, *Restart*), the variables and helpers from the
cells above are gone and the cells below would fail with `name '...' is not defined`. This cell loads them back
from `settings.json` and `nb_helpers.py`, without reinstalling or retraining. In an uninterrupted run it does nothing.

In [ ]:
if "video" not in globals():
  import json, os, shutil, subprocess
  from pathlib import Path
  # The output folder is the one whose settings.json was written last (OUTPUT_NAME in the settings).
  found = sorted(Path("/kaggle/working").glob("*/settings.json"), key=lambda p: p.stat().st_mtime)
  WORK = found[-1].parent if found else Path("/kaggle/working/hu_d03_rl")
  CODE = Path("/kaggle/temp/hu_d03_code")
  if not (WORK / "settings.json").exists() or not (CODE / "mjlab").exists():
    raise SystemExit(
      "This is a new Kaggle session: nothing is installed. Set TRAIN = False and run all cells; "
      "sections 1-5 reinstall, then the cells below use the checkpoints in /kaggle/working "
      "or in a previous version added under Input.")
  globals().update(json.loads((WORK / "settings.json").read_text()))
  if LINE_RANGE:
    LINE_RANGE = tuple(LINE_RANGE)
  LOG_ROOT = WORK / "logs" / "rsl_rl" / EXPERIMENT
  SCRIPTS = CODE / "hu-d03-rl" / "scripts"
  MOTION_FILE = WORK / "motions" / f"{RUN_NAME}_hu_d03.npz"
  exec((WORK / "nb_helpers.py").read_text())
  print("restored", RUN_NAME, "| runs:", sorted(p.name for p in LOG_ROOT.glob(f"*_{RUN_NAME}/")))

## 7. Training curves

Kaggle cannot show TensorBoard inline, so this plots the main curves from the event files.
The policy tracks the clip well when the episode length stays near the 10 s limit (no early terminations from
falling or drifting off the reference) and the `motion_body_pos` / `motion_global_root_pos` rewards are high.

In [ ]:
import matplotlib.pyplot as plt

def plot_curves():
  from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
  run = sorted(p for p in LOG_ROOT.glob(f"*_{RUN_NAME}/") if list(p.glob("events.*")))[-1]
  acc = EventAccumulator(str(run)); acc.Reload()
  tags = acc.Tags()["scalars"]
  wanted = [t for t in tags if t in ("Train/mean_reward", "Train/mean_episode_length")]
  wanted += [t for t in tags if t.startswith("Episode_Reward/")
             and any(k in t for k in ("motion_body_pos", "motion_body_ori", "motion_global_root_pos"))]
  fig, axes = plt.subplots(1, len(wanted), figsize=(4 * len(wanted), 3), squeeze=False)
  for ax, tag in zip(axes[0], wanted):
    ev = acc.Scalars(tag)
    ax.plot([e.step for e in ev], [e.value for e in ev])
    ax.set_title(tag.split("/")[-1], fontsize=9); ax.set_xlabel("iteration")
  fig.tight_layout()
  fig.savefig(run / "curves.png", dpi=120)
  plt.show()
  print("other tags:", [t for t in tags if t not in wanted][:40])

try:
  plot_curves()
except Exception as e:  # Never fail the run here.
  print("Could not plot curves:", e)

## 8. Video of the trained policy

For a medley, one video per clip (`videos/01_<clip>_mimic.mp4`, ...): each starts from the reference pose at the
clip's first frame and runs to its end. The green ghost is the reference; if the robot falls, the episode restarts at
the beginning of the medley, so a sudden jump to another motion in a video means it fell.

In [ ]:
def latest_checkpoint():
  runs = sorted(p for p in LOG_ROOT.glob(f"*_{RUN_NAME}/") if list(p.glob("model_*.pt")))
  ckpts = sorted(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
  print("checkpoint:", ckpts[-1])
  return ckpts[-1]

try:
  checkpoint = latest_checkpoint()
  if MEDLEY:
    # One video per clip of the medley, starting from the reference pose at the clip's first frame.
    # They are not embedded (the notebook would get too big): see the Output tab or the zip.
    from IPython.display import FileLink
    segments = json.loads((WORK / "motions" / f"{CLIP}_hu_d03.segments.json").read_text())
    for i, seg in enumerate(segments, 1):
      name = seg["clip"].removesuffix("_hu_d03")
      seconds = seg["end_s"] - seg["start_s"]
      print(f"\n{i}/{len(segments)} {name}: {seg['start_s']:.1f}-{seg['end_s']:.1f} s")
      out = video(f"videos/{i:02d}_{name}_mimic.mp4", f"--checkpoint-file {checkpoint}",
                  f"--start-seconds {seg['start_s']}", f"--seconds {seconds:.1f}", show=False)
      if out:
        display(FileLink(os.path.relpath(out, os.getcwd())))
  else:
    video(f"videos/{CLIP}_mimic.mp4", f"--checkpoint-file {checkpoint}", "--seconds 15")
except Exception as e:
  print("Could not record the policy video:", e)

## 9. Package the results

Writes `hu_d03_tracking_<run>.zip` with the newest checkpoint, the ONNX policy, the motion `.npz`, configs,
curves and videos.
Download it from the notebook's *Output* tab. The full `logs/` folder is in the output too, for resuming.

In [ ]:
run = sorted(p for p in LOG_ROOT.glob(f"*_{RUN_NAME}/") if list(p.glob("model_*.pt")))[-1]
pack = WORK / "export" / run.name
shutil.rmtree(pack, ignore_errors=True)
pack.mkdir(parents=True)
newest = max(run.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
shutil.copy(newest, pack)
for f in [*run.glob("*.onnx"), *run.glob("params/*"), *run.glob("curves.png")]:
  dst = pack / f.relative_to(run)
  dst.parent.mkdir(parents=True, exist_ok=True)
  shutil.copy(f, dst)
shutil.copy(MOTION_FILE, pack)
for f in (WORK / "motions").glob(f"{CLIP}_hu_d03.segments.json"):   # Medley only: where each dance starts.
  shutil.copy(f, pack)
if (WORK / "videos").exists():
  shutil.copytree(WORK / "videos", pack / "videos", dirs_exist_ok=True)
archive = shutil.make_archive(str(WORK / f"{EXPERIMENT}_{run.name}"), "zip", pack)
print("wrote", archive)
sh(f"ls -lR {pack} | head -40")

## 10. Download everything

Zips the whole `<OUTPUT_NAME>` folder (all checkpoints, ONNX, TensorBoard logs, `train.log`, videos) into
`<OUTPUT_NAME>_all.zip` and shows a link to download it straight to your computer. The link works in an
interactive session; for a saved version, download the zip from the *Output* tab instead.

In [ ]:
from IPython.display import FileLink

everything = WORK.parent / f"{WORK.name}_all"
Path(f"{everything}.zip").unlink(missing_ok=True)
shutil.make_archive(str(everything), "zip", WORK.parent, WORK.name)
size_mb = Path(f"{everything}.zip").stat().st_size / 1e6
print(f"wrote {everything}.zip ({size_mb:,.0f} MB)")
# FileLink paths are relative to the notebook's folder, /kaggle/working.
display(FileLink(os.path.relpath(f"{everything}.zip", os.getcwd())))